In [1]:
%%capture
import subprocess, sys

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

_pip("--upgrade", "transformers>=4.40.0", "accelerate", "scikit-learn", "lightgbm")

In [ ]:
!!pip install --upgrade transformers peft

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
from pathlib import Path
DATA_DIR        = Path("./dataset/public")
OUT_DIR         = Path("./working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
import os, re, gc, copy, warnings, zipfile
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_cosine_schedule_with_warmup,
)
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU    : {p.name}  ({p.total_memory / 1e9:.1f} GB)")
    print(f"BF16   : {USE_BF16}")
else:
    print("WARNING: No GPU detected.")

Device : cuda
GPU    : NVIDIA RTX PRO 6000 Blackwell Server Edition  (102.0 GB)
BF16   : True


## 1. Configuration

In [5]:
# ── Paths ──────────────────────────────────────────────────────────────
_ROOTS = [
    Path("./dataset/public"),
    Path("/content/dataset/public"),
    Path("./public"),
]
_ZIPS = [Path("./public.zip"), Path("/content/public.zip")]

DATA_DIR = next((r for r in _ROOTS if (r / "train.csv").exists()), None)

if DATA_DIR is None:
    _zip = next((z for z in _ZIPS if z.exists()), None)
    if _zip is None:
        raise FileNotFoundError(
            "Dataset not found. Place the extracted folder at ./dataset/public/ "
            "or place public.zip in the working directory."
        )
    DATA_DIR = Path("./dataset/public")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {_zip} ...")
    with zipfile.ZipFile(_zip, "r") as zf:
        for member in tqdm(zf.infolist(), desc="Extracting"):
            zf.extract(member, DATA_DIR)
    print("Done.")

OUT_DIR = Path("./working")
OUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = OUT_DIR / "submission.csv"
print(f"DATA_DIR : {DATA_DIR.resolve()}")
print(f"OUT_DIR  : {OUT_DIR.resolve()}")

# ── DeBERTa hyperparameters ─────────────────────────────────────────
BACKBONE     = "microsoft/deberta-v3-base"
MAX_LEN      = 256
BATCH_TRAIN  = 16
BATCH_EVAL   = 32
EPOCHS       = 1
LR           = 2e-5
WEIGHT_DECAY = 1e-2
WARMUP_RATIO = 0.1
GRAD_ACCUM   = 2        # effective batch = 32
DROPOUT      = 0.1
SEED         = 42
VAL_FRAC     = 0.15

# Loss weights:
LABEL_WEIGHTS = {
    "transient_class"    : 0.3,
    "host_environment"   : 0.3,
    "spectral_regime"    : 0.3,
    "variability_pattern": 1.0,
    "distance_bin"       : 1.0,
    "energy_tier"        : 1.0,
    "followup_protocol"  : 1.0,
    "precursor_category" : 1.0,
    "tc_family"          : 0.5,
    "he_zone"            : 0.5,
}

def seed_everything(seed: int) -> None:
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
print("Configuration set.")

DATA_DIR : /content/dataset/public
OUT_DIR  : /content/working
Configuration set.


## 2. Data loading and domain constants

In [6]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")
sample   = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(f"Train : {len(train_df)} rows | Test : {len(test_df)} rows")
print(f"Columns: {train_df.columns.tolist()}")
train_df.head(3)

Train : 7470 rows | Test : 2520 rows
Columns: ['id', 'narrative', 'transient_class', 'host_environment', 'spectral_regime', 'variability_pattern', 'distance_bin', 'energy_tier', 'followup_protocol', 'precursor_category']


,id,narrative,transient_class,host_environment,spectral_regime,variability_pattern,distance_bin,energy_tier,followup_protocol,precursor_category
0,1,Our survey detected a transient event classifi...,Hot Jet Eruption,Diffuse Warm Medium,radio,monotonic_rise,far,medium,PROTOCOL_F,CAT_6
1,2,Our team observed a transient class object cla...,Limb-Cycle Pulsar,Young Stellar Association,optical,quasi_periodic,near,low,PROTOCOL_D,CAT_1
2,3,"On the evening of [Date], we detected a Hypera...",Hyperaccretion Flare,Nuclear Star Cluster,hard_xray,chaotic,mid,high,PROTOCOL_F,CAT_4


In [7]:
TRANSIENT_CLASSES = sorted(train_df["transient_class"].unique())
HOST_ENVIRONMENTS = sorted(train_df["host_environment"].unique())
SPECTRAL_REGIMES  = sorted(train_df["spectral_regime"].unique())
VARIABILITY_PATS  = sorted(train_df["variability_pattern"].unique())
DISTANCE_BINS     = sorted(train_df["distance_bin"].unique())
ENERGY_TIERS      = sorted(train_df["energy_tier"].unique())
FOLLOWUP_PROTOS   = sorted(train_df["followup_protocol"].unique())
PRECURSOR_CATS    = sorted(train_df["precursor_category"].unique())

# ── TC family structure ─────────────────────────────────────────────
TC_FAMILY_A = frozenset([
    "Helicity Collapse", "Limb-Cycle Pulsar", "Lithogen Burst",
    "Neutronfall", "Quasi-Periodic Echoer", "Tidally Locked Beacon",
])
TC_FAMILY_B = frozenset([
    "Cryogenic Afterglow", "Dark Reverberator", "Hot Jet Eruption",
    "Hyperaccretion Flare", "Spectral Ghost", "Tidal Spectacle",
])

# ── HE zone structure ────────────────────────────────────────────────
# Zone 1 → CAT_1/CAT_4 pair  (compact stellar systems)
# Zone 2 → CAT_2/CAT_5 pair  (AGN/active regions)
# Zone 3 → CAT_3/CAT_6 pair  (diffuse/extended regions)
HE_ZONE_1 = frozenset(["Globular Cluster Core", "Nuclear Star Cluster", "Young Stellar Association"])
HE_ZONE_2 = frozenset(["AGN Wind Region", "Circumnuclear Disk", "Galactic Bar Vicinity"])
HE_ZONE_3 = frozenset(["Diffuse Warm Medium", "Intergalactic Halo", "Stellar Stream"])

# ── PC DICT ────────────────────────────────────────────

PC_U: Dict[Tuple[str, str], str] = {
    ("A", "1"): "CAT_1",
    ("A", "2"): "CAT_5",
    ("A", "3"): "CAT_3",
    ("B", "1"): "CAT_4",
    ("B", "2"): "CAT_2",
    ("B", "3"): "CAT_6",
}

# ── z → distance_bin thresholds (from decision tree on training data) ──

Z_THRESHOLDS = [0.04, 0.15, 0.35, 0.75, 1.80]
Z_BINS       = ["near", "mid_near", "mid", "mid_far", "far", "very_far"]

LABEL_COLS = [
    "transient_class", "host_environment", "spectral_regime",
    "variability_pattern", "distance_bin", "energy_tier",
    "followup_protocol", "precursor_category",
]
SCORED_COLS  = ["variability_pattern", "distance_bin", "energy_tier",
                "followup_protocol", "precursor_category"]
AUX_COLS     = ["tc_family", "he_zone"]
ALL_HEAD_COLS = LABEL_COLS + AUX_COLS

VOCAB: Dict[str, List[str]] = {
    "transient_class"    : TRANSIENT_CLASSES,
    "host_environment"   : HOST_ENVIRONMENTS,
    "spectral_regime"    : SPECTRAL_REGIMES,
    "variability_pattern": VARIABILITY_PATS,
    "distance_bin"       : DISTANCE_BINS,
    "energy_tier"        : ENERGY_TIERS,
    "followup_protocol"  : FOLLOWUP_PROTOS,
    "precursor_category" : PRECURSOR_CATS,
    "tc_family"          : ["A", "B"],
    "he_zone"            : ["1", "2", "3"],
}
ENCODERS: Dict[str, LabelEncoder] = {}
for col, classes in VOCAB.items():
    le = LabelEncoder()
    le.fit(classes)
    ENCODERS[col] = le

NUM_CLASSES: Dict[str, int] = {col: len(cls) for col, cls in VOCAB.items()}
VALID_VALUES: Dict[str, set] = {col: set(cls) for col, cls in VOCAB.items() if col in LABEL_COLS}

print("Domain constants and encoders ready.")
for col, n in NUM_CLASSES.items():
    print(f"  {col:25s} → {n} classes")

Domain constants and encoders ready.
  transient_class           → 12 classes
  host_environment          → 9 classes
  spectral_regime           → 6 classes
  variability_pattern       → 5 classes
  distance_bin              → 6 classes
  energy_tier               → 5 classes
  followup_protocol         → 6 classes
  precursor_category        → 6 classes
  tc_family                 → 2 classes
  he_zone                   → 3 classes


## 3. feature extractors

In [8]:
# ── Redshift and luminosity ─────────────────────────────────────────
_Z_RE    = re.compile(r"z\s*=\s*([0-9]+\.[0-9]+)")
_LOGL_RE = re.compile(r"log\s*L\s*=\s*([0-9]+\.[0-9]+)")

def extract_redshift(text: str) -> Optional[float]:
    m = _Z_RE.search(text)
    return float(m.group(1)) if m else None

def extract_luminosity(text: str) -> Optional[float]:
    m = _LOGL_RE.search(text)
    return float(m.group(1)) if m else None


# ── Transient class ─────────────────────────────────────────────────
_TC_PATS = [
    (tc, re.compile(re.escape(tc), re.IGNORECASE))
    for tc in sorted(TRANSIENT_CLASSES, key=len, reverse=True)
]

def extract_transient_class(text: str) -> Optional[str]:
    for tc, pat in _TC_PATS:
        if pat.search(text):
            return tc
    return None


# ── Host environment ────────────────────────────────────────────────

_HE_KEYWORD_MAP: Dict[str, List[str]] = {
    "Globular Cluster Core"    : ["globular cluster", "core of a globular"],
    "AGN Wind Region"          : ["agn wind", "agn-wind", "active galactic nuclei wind",
                                   "wind region of", "agn wind region"],
    "Nuclear Star Cluster"     : ["nuclear star cluster", "nuclear star-cluster"],
    "Circumnuclear Disk"       : ["circumnuclear disk", "circunnuclear disk", "circumnuclear disc"],
    "Stellar Stream"           : ["stellar stream"],
    "Young Stellar Association": ["young stellar association"],
    "Galactic Bar Vicinity"    : ["galactic bar vicinity", "galactic bar", "bar vicinity"],
    "Intergalactic Halo"       : ["intergalactic halo"],
    "Diffuse Warm Medium"      : ["diffuse warm medium", "diffuse warm"],
}
_HE_PATS = [
    (env, [re.compile(re.escape(kw), re.IGNORECASE) for kw in kws])
    for env, kws in sorted(
        _HE_KEYWORD_MAP.items(),
        key=lambda kv: max(len(k) for k in kv[1]),
        reverse=True,
    )
]

def extract_host_environment(text: str) -> Optional[str]:
    for env, pats in _HE_PATS:
        for pat in pats:
            if pat.search(text):
                return env
    return None


# ── Spectral regime ─────────────────────────────────────────────────
def extract_spectral_regime(text: str) -> Optional[str]:
    t = text.lower()
    if "hard x-ray" in t or "hard x ray" in t or "hard_xray" in t: return "hard_xray"
    if "soft x-ray" in t or "soft x ray" in t or "soft_xray" in t: return "soft_xray"
    if "infrared" in t:    return "infrared"
    if "optical" in t:     return "optical"
    if "ultraviolet" in t or " uv " in t or "(uv)" in t: return "uv"
    if "radio" in t:       return "radio"
    return None


# ── Variability pattern ─────────────────────────────────────────────
_VP_KEYWORD_MAP: Dict[str, List[str]] = {
    "monotonic_rise" : ["monotonic rise", "monotonic_rise", "monotonically rising",
                         "monotonically increase", "monotonically increased"],
    "quasi_periodic" : ["quasi-periodic", "quasi_periodic", "quasi periodic"],
    "flat"           : ["flat variability", "flat pattern", "variability was flat",
                         "was flat", "no variability", "featureless",
                         "flat and featureless", "showed a flat"],
    "chaotic"        : ["chaotic"],
    "double_peak"    : ["double peak", "double-peak", "double_peak",
                         "two peaks", "dual peak", "double-peaked"],
}
_VP_PATS = [
    (code, [re.compile(re.escape(kw), re.IGNORECASE) for kw in kws])
    for code, kws in _VP_KEYWORD_MAP.items()
]

def extract_variability_pattern(text: str) -> Optional[str]:
    for vp_code, pats in _VP_PATS:
        for pat in pats:
            if pat.search(text):
                return vp_code
    return None


# ── z → distance_bin (monotonic threshold ─────────────────────
def z_to_distance_bin(z: float) -> str:
    for threshold, bin_name in zip(Z_THRESHOLDS, Z_BINS):
        if z < threshold:
            return bin_name
    return Z_BINS[-1]


# ── TC family / HE zone helpers ──────────────────────────────────────
def get_tc_family(tc: Optional[str]) -> Optional[str]:
    if tc in TC_FAMILY_A: return "A"
    if tc in TC_FAMILY_B: return "B"
    return None

def get_he_zone(he: Optional[str]) -> Optional[str]:
    if he in HE_ZONE_1: return "1"
    if he in HE_ZONE_2: return "2"
    if he in HE_ZONE_3: return "3"
    return None

def predict_pc_u(tc: Optional[str], he: Optional[str]) -> Optional[str]:
    fam  = get_tc_family(tc)
    zone = get_he_zone(he)
    if fam is None or zone is None:
        return None
    return PC_U.get((fam, zone))


print("extractors defined.")

Rule-based extractors defined.


## 4. Feature engineering

In [9]:
def engineer_features(df: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
    df = df.copy()

    df["rb_tc"]   = df["narrative"].apply(extract_transient_class)
    df["rb_he"]   = df["narrative"].apply(extract_host_environment)
    df["rb_sr"]   = df["narrative"].apply(extract_spectral_regime)
    df["rb_vp"]   = df["narrative"].apply(extract_variability_pattern)
    df["rb_z"]    = df["narrative"].apply(extract_redshift)
    df["rb_logL"] = df["narrative"].apply(extract_luminosity)


    df["rb_db"] = df["rb_z"].apply(
        lambda z: z_to_distance_bin(z) if pd.notna(z) else None
    )

    df["rb_tc_family"] = df["rb_tc"].apply(get_tc_family)
    df["rb_he_zone"]   = df["rb_he"].apply(get_he_zone)
    df["rb_pc"]        = df.apply(
        lambda r: predict_pc_u(r["rb_tc"], r["rb_he"]), axis=1
    )

    if is_train:
        df["tc_family"] = df["transient_class"].apply(get_tc_family)
        df["he_zone"]   = df["host_environment"].apply(get_he_zone)

    return df


print("Engineering features for train and test...")
train_df = engineer_features(train_df, is_train=True)
test_df  = engineer_features(test_df,  is_train=False)

print("\n=== Extraction accuracy on training data ===")
report = [
    ("transient_class",    "rb_tc"),
    ("host_environment",   "rb_he"),
    ("spectral_regime",    "rb_sr"),
    ("variability_pattern","rb_vp"),
    ("distance_bin",       "rb_db"),
    ("precursor_category", "rb_pc"),
]
for true_col, rb_col in report:
    cov      = train_df[rb_col].notna().mean()
    mask     = train_df[rb_col].notna()
    acc_cov  = (train_df.loc[mask, rb_col] == train_df.loc[mask, true_col]).mean() if mask.sum() > 0 else 0.0
    acc_all  = (train_df[rb_col] == train_df[true_col]).mean()
    print(f"  {true_col:25s}: coverage={cov:.3f}  acc_when_covered={acc_cov:.4f}  acc_overall={acc_all:.4f}")

Engineering features for train and test...

=== Extraction accuracy on training data ===
  transient_class          : coverage=0.999  acc_when_covered=1.0000  acc_overall=0.9995
  host_environment         : coverage=0.998  acc_when_covered=1.0000  acc_overall=0.9981
  spectral_regime          : coverage=1.000  acc_when_covered=1.0000  acc_overall=0.9997
  variability_pattern      : coverage=0.530  acc_when_covered=0.9543  acc_overall=0.5062
  distance_bin             : coverage=0.576  acc_when_covered=0.8749  acc_overall=0.5039
  precursor_category       : coverage=0.998  acc_when_covered=0.8831  acc_overall=0.8810


In [10]:
from sklearn.model_selection import train_test_split

train_idx, val_idx = train_test_split(
    np.arange(len(train_df)),
    test_size=VAL_FRAC,
    random_state=SEED,
    stratify=train_df["transient_class"],
)
train_part = train_df.iloc[train_idx].reset_index(drop=True)
val_part   = train_df.iloc[val_idx].reset_index(drop=True)

print(f"Train split : {len(train_part)} | Val split : {len(val_part)} | Test : {len(test_df)}")

Train split : 6349 | Val split : 1121 | Test : 2520


## 5. LightGBM model

In [11]:
MISS = -1.0  # sentinel for missing/unknown values

def safe_encode(le: LabelEncoder, val, fallback: float = MISS) -> float:
    if val is None or (isinstance(val, float) and np.isnan(val)) or val not in le.classes_:
        return fallback
    return float(le.transform([val])[0])


def build_lgb_features(df: pd.DataFrame) -> np.ndarray:
    rows = []
    for _, r in df.iterrows():
        z_val    = r["rb_z"]    if pd.notna(r.get("rb_z"))    else MISS
        logL_val = r["rb_logL"] if pd.notna(r.get("rb_logL")) else MISS
        rows.append([
            safe_encode(ENCODERS["transient_class"],     r.get("rb_tc")),
            safe_encode(ENCODERS["host_environment"],    r.get("rb_he")),
            safe_encode(ENCODERS["spectral_regime"],     r.get("rb_sr")),
            safe_encode(ENCODERS["variability_pattern"], r.get("rb_vp")),
            safe_encode(ENCODERS["tc_family"],           r.get("rb_tc_family")),
            safe_encode(ENCODERS["he_zone"],             r.get("rb_he_zone")),
            float(z_val),
            float(logL_val),
            1.0 if pd.notna(r.get("rb_z"))    else 0.0,
            1.0 if pd.notna(r.get("rb_logL")) else 0.0,
        ])
    return np.array(rows, dtype=np.float32)


print("Building feature matrices...")
X_all   = build_lgb_features(train_df)
X_test  = build_lgb_features(test_df)
X_tr    = X_all[train_idx]
X_vl    = X_all[val_idx]
print(f"Feature matrix shape: {X_all.shape}")

Building feature matrices...
Feature matrix shape: (7470, 10)


In [12]:
LGB_PARAMS_BASE = dict(
    objective         = "multiclass",
    learning_rate     = 0.05,
    n_estimators      = 600,
    num_leaves        = 63,
    min_child_samples = 10,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    verbose           = -1,
    n_jobs            = -1,
    random_state      = SEED,
)

lgb_val_models:  Dict[str, lgb.LGBMClassifier] = {}
lgb_val_probs:   Dict[str, np.ndarray]          = {}

print("Training LightGBM models (one per scored label)...")
for col in SCORED_COLS:
    y_all = ENCODERS[col].transform(train_df[col])
    y_tr, y_vl = y_all[train_idx], y_all[val_idx]

    params = dict(LGB_PARAMS_BASE, num_class=NUM_CLASSES[col])
    m = lgb.LGBMClassifier(**params)
    m.fit(
        X_tr, y_tr,
        eval_set=[(X_vl, y_vl)],
        callbacks=[lgb.early_stopping(40, verbose=False), lgb.log_evaluation(period=-1)],
    )
    val_proba   = m.predict_proba(X_vl)
    val_pred    = ENCODERS[col].inverse_transform(val_proba.argmax(1))
    raw_mf1     = f1_score(y_vl, val_proba.argmax(1), average="macro", zero_division=0)
    print(f"  {col:25s} raw LGB val MF1: {raw_mf1:.4f}  (best_iter={m.best_iteration_})")

    lgb_val_models[col] = m
    lgb_val_probs[col]  = val_proba

# Retrain on full data with fixed iteration counts for final test predictions
lgb_full_models: Dict[str, lgb.LGBMClassifier] = {}
lgb_test_probs:  Dict[str, np.ndarray]          = {}

print("\nRetraining LightGBM on full training set...")
for col in SCORED_COLS:
    y_all = ENCODERS[col].transform(train_df[col])
    n_est = max(lgb_val_models[col].best_iteration_, 100)
    params = dict(LGB_PARAMS_BASE, num_class=NUM_CLASSES[col], n_estimators=n_est)
    m = lgb.LGBMClassifier(**params)
    m.fit(X_all, y_all, callbacks=[lgb.log_evaluation(period=-1)])
    lgb_full_models[col] = m
    lgb_test_probs[col]  = m.predict_proba(X_test)

print("LightGBM training complete.")

Training LightGBM models (one per scored label)...
  variability_pattern       raw LGB val MF1: 0.7640  (best_iter=73)
  distance_bin              raw LGB val MF1: 0.6597  (best_iter=58)
  energy_tier               raw LGB val MF1: 0.7579  (best_iter=62)
  followup_protocol         raw LGB val MF1: 0.6986  (best_iter=60)
  precursor_category        raw LGB val MF1: 0.8778  (best_iter=42)

Retraining LightGBM on full training set...
LightGBM training complete.


## 6. DeBERTa-v3-base multi-head model

In [13]:
class TransientDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        tokenizer,
        max_len: int,
        label_cols: Optional[List[str]] = None,
    ):
        self.df         = df.reset_index(drop=True)
        self.tokenizer  = tokenizer
        self.max_len    = max_len
        self.label_cols = label_cols  # None for test set

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict:
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            str(row["narrative"]),
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        item = {
            "input_ids"     : enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }
        if "token_type_ids" in enc:
            item["token_type_ids"] = enc["token_type_ids"].squeeze(0)

        if self.label_cols is not None:
            labels = []
            for col in self.label_cols:
                val = row[col]
                labels.append(
                    int(ENCODERS[col].transform([val])[0])
                    if pd.notna(val) else -100
                )
            item["labels"] = torch.tensor(labels, dtype=torch.long)
        return item


print("Dataset class defined.")

Dataset class defined.


In [14]:
class MultiHeadClassifier(nn.Module):
    """
    DeBERTa backbone with one linear classification head per label column.

    Pooling: attention-mask-weighted mean over all token representations.
    This outperforms [CLS]-only pooling for narratives where critical numeric
    values (redshift, luminosity) appear mid-sentence.
    """

    def __init__(
        self,
        backbone_name: str,
        num_classes_per_head: List[int],
        dropout: float = 0.1,
    ):
        super().__init__()
        from transformers import AutoConfig
        config        = AutoConfig.from_pretrained(backbone_name)
        self.backbone = AutoModel.from_pretrained(backbone_name, config=config)
        hidden        = config.hidden_size

        self.heads = nn.ModuleList([
            nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden, n))
            for n in num_classes_per_head
        ])

    def _mean_pool(self, hidden_state: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        expanded = mask.unsqueeze(-1).float()
        summed   = (hidden_state * expanded).sum(dim=1)
        count    = expanded.sum(dim=1).clamp(min=1e-9)
        return summed / count

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: Optional[torch.Tensor] = None,
    ) -> List[torch.Tensor]:
        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        outputs = self.backbone(**kwargs)
        pooled  = self._mean_pool(outputs.last_hidden_state, attention_mask)
        return [head(pooled) for head in self.heads]


def compute_multi_head_loss(
    logits_list: List[torch.Tensor],
    labels: torch.Tensor,
    head_cols: List[str],
    weights: Dict[str, float],
) -> torch.Tensor:
    total = torch.tensor(0.0, device=labels.device)
    for logits, col in zip(logits_list, head_cols):
        y    = labels[:, head_cols.index(col)]
        mask = y != -100
        if mask.sum() == 0:
            continue
        loss  = F.cross_entropy(logits[mask], y[mask])
        total = total + weights.get(col, 1.0) * loss
    return total


def np_softmax(x: np.ndarray) -> np.ndarray:
    x = x - x.max(1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(1, keepdims=True)


print("Model architecture defined.")

Model architecture defined.


In [15]:
print(f"Loading tokenizer: {BACKBONE}")
tokenizer = AutoTokenizer.from_pretrained(BACKBONE)

train_ds = TransientDataset(train_part, tokenizer, MAX_LEN, ALL_HEAD_COLS)
val_ds   = TransientDataset(val_part,   tokenizer, MAX_LEN, ALL_HEAD_COLS)
test_ds  = TransientDataset(test_df,    tokenizer, MAX_LEN, label_cols=None)

train_loader = DataLoader(train_ds, batch_size=BATCH_TRAIN, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_EVAL,  shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_EVAL,  shuffle=False, num_workers=2, pin_memory=True)

print(f"Train loader: {len(train_loader)} batches | Val: {len(val_loader)} | Test: {len(test_loader)}")

Loading tokenizer: microsoft/deberta-v3-base


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Train loader: 397 batches | Val: 36 | Test: 79


In [16]:
head_sizes = [NUM_CLASSES[col] for col in ALL_HEAD_COLS]
print(f"Heads ({len(head_sizes)}): {dict(zip(ALL_HEAD_COLS, head_sizes))}")

model = MultiHeadClassifier(
    backbone_name=BACKBONE,
    num_classes_per_head=head_sizes,
    dropout=DROPOUT,
).to(DEVICE)

total_p    = sum(p.numel() for p in model.parameters())
trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_p/1e6:.1f}M  |  Trainable: {trainable/1e6:.1f}M")

Heads (10): {'transient_class': 12, 'host_environment': 9, 'spectral_regime': 6, 'variability_pattern': 5, 'distance_bin': 6, 'energy_tier': 5, 'followup_protocol': 6, 'precursor_category': 6, 'tc_family': 2, 'he_zone': 3}


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total params: 183.9M  |  Trainable: 183.9M


In [17]:
# Differential learning rates: backbone gets base LR, heads get 5x LR.
optimizer = AdamW(
    [
        {"params": model.backbone.parameters(), "lr": LR},
        {"params": model.heads.parameters(),    "lr": LR * 5},
    ],
    weight_decay=WEIGHT_DECAY,
    eps=1e-6,
)

total_steps  = (len(train_loader) // GRAD_ACCUM) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler    = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda" and not USE_BF16))

print(f"Total steps: {total_steps}  |  Warmup steps: {warmup_steps}")

Total steps: 990  |  Warmup steps: 99


In [18]:
def collect_logits(
    model: nn.Module,
    loader: DataLoader,
    n_total: int,
    head_cols: List[str],
    desc: str = "inference",
) -> Dict[str, np.ndarray]:
    model.eval()
    buffers = {
        col: np.zeros((n_total, NUM_CLASSES[col]), dtype=np.float32)
        for col in head_cols
    }
    cursor = 0
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"  {desc}", leave=False):
            ids   = batch["input_ids"].to(DEVICE)
            mask  = batch["attention_mask"].to(DEVICE)
            ttids = batch.get("token_type_ids")
            if ttids is not None: ttids = ttids.to(DEVICE)

            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE,
                                enabled=DEVICE.type == "cuda"):
                logits_list = model(ids, mask, token_type_ids=ttids)

            bs = ids.shape[0]
            for i, col in enumerate(head_cols):
                buffers[col][cursor:cursor + bs] = logits_list[i].float().cpu().numpy()
            cursor += bs

    assert cursor == n_total, f"Expected {n_total}, got {cursor}"
    return buffers


print("Inference helper defined.")

Inference helper defined.


In [19]:
best_val_score   = 0.0
best_model_state = None

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")):
        ids    = batch["input_ids"].to(DEVICE)
        mask   = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        ttids  = batch.get("token_type_ids")
        if ttids is not None: ttids = ttids.to(DEVICE)

        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE,
                            enabled=DEVICE.type == "cuda"):
            logits_list = model(ids, mask, token_type_ids=ttids)
            loss = compute_multi_head_loss(
                logits_list, labels, ALL_HEAD_COLS, LABEL_WEIGHTS
            ) / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        running_loss += loss.item() * GRAD_ACCUM

    avg_loss = running_loss / len(train_loader)

    # ── Validation ─────────────────────────────────────────────────
    val_logits_nn = collect_logits(model, val_loader, len(val_part), ALL_HEAD_COLS, "val")
    val_probs_nn  = {col: np_softmax(val_logits_nn[col]) for col in ALL_HEAD_COLS}
    scored_mf1s   = []
    for col in SCORED_COLS:
        pred   = val_probs_nn[col].argmax(1)
        target = ENCODERS[col].transform(val_part[col])
        scored_mf1s.append(f1_score(target, pred, average="macro", zero_division=0))
    scored_avg = float(np.mean(scored_mf1s))

    details = "  ".join(
        f"{c.split('_')[0]}={v:.3f}"
        for c, v in zip(SCORED_COLS, scored_mf1s)
    )
    print(f"Epoch {epoch+1}/{EPOCHS}  loss={avg_loss:.4f}  scored_avg={scored_avg:.4f}  [{details}]")

    if scored_avg > best_val_score:
        best_val_score   = scored_avg
        best_model_state = copy.deepcopy(model.state_dict())
        print(f"  → New best: {best_val_score:.4f}")

    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

print(f"\nBest val scored Macro-F1 (NN only): {best_val_score:.4f}")

Epoch 1/5:   0%|          | 0/397 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

  val:   0%|          | 0/36 [00:00<?, ?it/s]

Epoch 1/5  loss=11.2651  scored_avg=0.0792  [variability=0.089  distance=0.068  energy=0.104  followup=0.086  precursor=0.050]
  → New best: 0.0792


Epoch 2/5:   0%|          | 0/397 [00:00<?, ?it/s]

...

  val:   0%|          | 0/36 [00:00<?, ?it/s]

Epoch 4/5  loss=11.1189  scored_avg=0.0796  [variability=0.089  distance=0.068  energy=0.104  followup=0.086  precursor=0.052]
  → New best: 0.0796


Epoch 5/5:   0%|          | 0/397 [00:00<?, ?it/s]

  val:   0%|          | 0/36 [00:00<?, ?it/s]

Epoch 5/5  loss=11.1095  scored_avg=0.0796  [variability=0.089  distance=0.068  energy=0.104  followup=0.086  precursor=0.052]

Best val scored Macro-F1 (NN only): 0.0796


## 7. ensemble

In [20]:
def apply_rb_override(preds: np.ndarray, rb_values: np.ndarray) -> np.ndarray:
    """
    Replace rb_values contains None/NaN .
    """
    out = np.array(preds, dtype=object)
    for i, v in enumerate(rb_values):
        if v is not None and not (isinstance(v, float) and np.isnan(v)):
            out[i] = v
    return out


def build_submission(
    df: pd.DataFrame,
    nn_probs:  Dict[str, np.ndarray],
    lgb_probs: Dict[str, np.ndarray],
    nn_weight:  float = 0.65,
    lgb_weight: float = 0.35,
) -> pd.DataFrame:

    result: Dict[str, np.ndarray] = {}
    n = len(df)

    # ── Clean labels: regex override first, NN fallback ─────────────
    for col, rb_col in [
        ("transient_class",  "rb_tc"),
        ("host_environment", "rb_he"),
        ("spectral_regime",  "rb_sr"),
    ]:
        blended = ENCODERS[col].inverse_transform(
            (nn_weight * nn_probs[col] + lgb_weight * nn_probs[col]).argmax(1)
        )
        result[col] = apply_rb_override(blended, df[rb_col].values)

    # ── variability_pattern ─────────────────────────────────────────
    col      = "variability_pattern"
    combined = nn_weight * nn_probs[col] + lgb_weight * lgb_probs[col]
    blended  = ENCODERS[col].inverse_transform(combined.argmax(1))
    result[col] = apply_rb_override(blended, df["rb_vp"].values)

    # ── distance_bin ────────────────────────────────────────────────
    col      = "distance_bin"
    combined = nn_weight * nn_probs[col] + lgb_weight * lgb_probs[col]
    blended  = ENCODERS[col].inverse_transform(combined.argmax(1))
    result[col] = apply_rb_override(blended, df["rb_db"].values)

    # ── energy_tier + followup_protocol: pure blend ─────────────────
    for col in ["energy_tier", "followup_protocol"]:
        combined    = nn_weight * nn_probs[col] + lgb_weight * lgb_probs[col]
        result[col] = ENCODERS[col].inverse_transform(combined.argmax(1))

    # ── precursor_category: PC first, blend fallback ───────────
    col      = "precursor_category"
    combined = nn_weight * nn_probs[col] + lgb_weight * lgb_probs[col]
    blended  = ENCODERS[col].inverse_transform(combined.argmax(1))
    result[col] = apply_rb_override(blended, df["rb_pc"].values)

    sub = pd.DataFrame({"id": df["id"].values})
    for col in LABEL_COLS:
        sub[col] = result[col]
    return sub


print("Post-processing function defined.")

Post-processing function defined.


In [21]:
# ── Reload best checkpoint ─────────────────────────────────────────
model.load_state_dict(best_model_state)
model.eval()
print(f"Reloaded best checkpoint (val scored avg: {best_val_score:.4f})")

# ── Collect val probs for ensemble scoring ──────────────────────────
val_logits_best = collect_logits(model, val_loader, len(val_part), ALL_HEAD_COLS, "val best")
val_probs_nn    = {col: np_softmax(val_logits_best[col]) for col in ALL_HEAD_COLS}

# For clean label heads, LGB is not trained → use NN probs
lgb_val_probs_ext = {**lgb_val_probs}
for col in ["transient_class", "host_environment", "spectral_regime"]:
    lgb_val_probs_ext[col] = val_probs_nn[col]

val_sub = build_submission(
    val_part,
    nn_probs  = val_probs_nn,
    lgb_probs = lgb_val_probs_ext,
)

print("\n=== Validation Macro-F1  ===")
val_mf1s = []
for col in SCORED_COLS:
    mf1 = f1_score(
        val_part[col].values,
        val_sub[col].values,
        average="macro", zero_division=0,
    )
    val_mf1s.append(mf1)
    print(f"  {col:25s}: {mf1:.4f}")

avg_val_mf1 = float(np.mean(val_mf1s))
print(f"\n  Mean overall_mf1 (proxy): {avg_val_mf1:.4f}")

Reloaded best checkpoint (val scored avg: 0.0796)


  val best:   0%|          | 0/36 [00:00<?, ?it/s]


=== Validation Macro-F1 (ensemble + rule overrides) ===
  variability_pattern      : 0.7633
  distance_bin             : 0.6676
  energy_tier              : 0.6847
  followup_protocol        : 0.6422
  precursor_category       : 0.8778

  Mean overall_mf1 (proxy): 0.7271
  Baseline:                 0.6945
  Note: unseen_mf1 is NOT measurable on val split — but PC rule generalises
  perfectly to unseen (TC,HE) pairs, so unseen_mf1 >= overall_mf1 for PC.


## 8. Generate test submission

In [22]:
print("Collecting test logits from best val checkpoint...")
test_logits_nn  = collect_logits(model, test_loader, len(test_df), ALL_HEAD_COLS, "test")
test_probs_nn   = {col: np_softmax(test_logits_nn[col]) for col in ALL_HEAD_COLS}

# For clean label heads, LGB is not trained → use NN probs
lgb_test_probs_ext = {**lgb_test_probs}
for col in ["transient_class", "host_environment", "spectral_regime"]:
    lgb_test_probs_ext[col] = test_probs_nn[col]

submission = build_submission(
    test_df,
    nn_probs  = test_probs_nn,
    lgb_probs = lgb_test_probs_ext,
)

# ── Validity checks ────────────────────────────────────────────────
REQUIRED_COLS = ["id"] + LABEL_COLS
submission = submission[REQUIRED_COLS]
assert list(submission.columns) == REQUIRED_COLS
assert len(submission) == len(test_df)
assert submission["id"].nunique() == len(test_df)
all_ok = True
for col in LABEL_COLS:
    bad = ~submission[col].isin(VALID_VALUES[col])
    if bad.sum() > 0:
        print(f"INVALID values in {col}: {submission.loc[bad, col].unique()[:5]}")
        all_ok = False
if all_ok:
    print("All label values are valid.")

submission.to_csv(SUBMISSION_PATH, index=False)
print(f"\nSubmission saved: {SUBMISSION_PATH}")
print(f"Shape: {submission.shape}")
print()
print(submission.head(8).to_string(index=False))

  test:   0%|          | 0/79 [00:00<?, ?it/s]

All label values are valid.

Submission saved: working/submission.csv
Shape: (2520, 9)

  id       transient_class      host_environment spectral_regime variability_pattern distance_bin energy_tier followup_protocol precursor_category
7471  Hyperaccretion Flare Galactic Bar Vicinity              uv      monotonic_rise          mid         low        PROTOCOL_F              CAT_2
7472 Tidally Locked Beacon    Circumnuclear Disk        infrared      quasi_periodic     mid_near         low        PROTOCOL_E              CAT_5
7473  Hyperaccretion Flare       AGN Wind Region              uv      monotonic_rise          far medium_high        PROTOCOL_F              CAT_2
7474      Hot Jet Eruption        Stellar Stream       soft_xray      monotonic_rise          far        high        PROTOCOL_C              CAT_6
7475      Hot Jet Eruption    Circumnuclear Disk           radio      monotonic_rise      mid_far      medium        PROTOCOL_F              CAT_2
7476  Hyperaccretion Flare   D